Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter
from tools.coverage_functions import plot_time_series, plot_time_series_subset
from tools.labeling_functions import fully_relabel_and_consolidate, plot_dish_time_series, rename_items, rename_items_by_modifications

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')
    %store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
    %store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()
    %store restaurants_by_4m_coverage

loc_id = 'LBZEEFSBJNB3Z'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
static_data['locations'].sort_index()

In [ ]:
print(df_uncleaned.query('item_type != "Drink"')['item_name'].value_counts().to_string())

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.query('item_name.str.contains("Wake")').query('item_modifications.str.contains("Vegan")')['item_quantity'].sum()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Wake")').query('item_modifications.str.contains("Vegan")')['item_quantity'].sum()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plt.show()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='D', 
                 subset=True)
plt.show()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned.query('item_name.str.contains("Veggie Sausage")'), 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plt.show()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned.query('item_name.str.contains("Veggie Sausage")'), 
                 before_after_details_true, 
                 freq='D', 
                 subset=True)
plt.show()

In [ ]:
plot_time_series(loc_id, 
                 df_uncleaned.query('item_name.str.contains("Veggie-Lito")'), 
                 before_after_details_true, 
                 freq='D', 
                 subset=False)
plt.show()

In [ ]:
sales_and_menu_data[loc_id].index

In [ ]:
time_differences_details[loc_id]

In [ ]:
df_uncleaned['dish_category'].value_counts()

In [ ]:
df_uncleaned.query('item_name.str.contains("The Dipsticks")')['item_modifications'].value_counts()

In [ ]:
drink_cat = ["Coffee & Tea", "Juice", "Soda", "Water", "Alcohol"]
drink_type = ["Drink"]
plot_dish_time_series(df_uncleaned.query('~dish_category.isin(@drink_cat) and ~item_type.isin(@drink_type) and item_modifications.str.contains("Vegan Egg|Vegan Sausage|Vegan Cheese|Vegan Bacon")'), loc_id, before_after_details_true)